# Predicción del fútbol uruguayo: notebook común y entrega de Naive Bayes

**Ejecución desde cero:** usar Python 3.12 y scikit-learn 1.9, instalar
`requirements.txt` y ejecutar las celdas en orden desde la raíz del proyecto.
Las únicas entradas necesarias para NB son el ZIP original
`data/raw/futbol_uruguayo.zip` y los módulos de `src/`: no requiere ningún
CSV, modelo ni manifiesto precalculado en `results/`.

Las secciones 7–10 reproducen la selección temporal de NB propio y CategoricalNB:
30 ajustes de suavizado sobre seis tasas y 42 de atributos con suavizado fijo,
todos hasta 2023. La sección 14 ajusta los dos NB finales con diez atributos
y el baseline, evalúa los mismos partidos de 2024–2025 y muestra predicciones
y análisis. No se amplía la búsqueda ni se selecciona con test.

Se conserva el trabajo de árboles y Random Forest: sus tablas y figuras
históricas se muestran si existen. Sus celdas finales siguen desactivadas
(`RUN_FINAL_TEST = False`) hasta integrar su evaluación definitiva.
`python scripts/verify_final_nb.py` ejecuta todo el notebook con esas celdas
desactivadas en una copia temporal sin resultados previos y guarda evidencia en
`results/nb_delivery/`. El informe común es `informe.tex`.


## 1. Configuración

Entorno de la consigna: Python 3.12 y scikit-learn 1.9. Los módulos de `src/`
comparten la política de datos, historiales y validación. La semilla es 42.


In [ ]:

# Configuración inicial y utilidades

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

import sklearn
if sys.version_info[:2] != (3, 12) or not sklearn.__version__.startswith('1.9.'):
    raise RuntimeError('Este notebook requiere Python 3.12 y scikit-learn 1.9.')
print('Entorno:', sys.version.split()[0], 'scikit-learn', sklearn.__version__)
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'requirements.txt').is_file() and (p / 'src').is_dir())
sys.path.insert(0, str(ROOT / 'src'))

from features import (  # noqa: E402
    NUMERIC_FEATURES,
    build_causal_match_features,
    load_clean_matches,
)
from baseline import TenYearWinRateClassifier  # noqa: E402
from id3 import ID3  # noqa: E402
from naive_bayes import MEstimateCategoricalNB  # noqa: E402
from preprocessing import MixedTypeDiscretizer  # noqa: E402
from baseline import BASELINE_FEATURES  # noqa: E402
from evaluation import (  # noqa: E402
    temporal_holdout, make_temporal_folds,
    new_discretizer, evaluate_temporal_cv,
)

RAW = ROOT / 'data/raw/futbol_uruguayo.zip'
CLASSES = ['E', 'L', 'V']

# discretización
BIN_ORDER = ['baja', 'media', 'alta']
RANDOM_STATE = 42
pd.set_option('display.max_columns', 30)


def entropia(series: pd.Series) -> float:
    probs = series.value_counts(normalize=True)
    return float(-(probs * np.log2(probs)).sum())

## 2. Datos: lectura, auditoría y limpieza del ZIP

`load_clean_matches(..., through_year=2023)` separa por fecha antes de interpretar
marcadores o construir etiquetas y aplica la política de `docs/data_policy.md` antes de crear
etiquetas o historiales: elimina repeticiones exactas, pone en cuarentena todas
las variantes de una fecha/par de equipos ambiguo (incluida localía invertida)
y excluye los registros `full_time=E/P`, cuyos goles a los 90 minutos no están
separados. No elige un marcador entre variantes ni reconstruye goles.

En los registros `F` conservados, `winner` es `L` si `gh > ga`, `V` si `gh < ga`
y `E` si son iguales. `full_time=E` significa prórroga, no la etiqueta empate.
Los goles del partido no son entradas. La auditoría reproducible, con líneas
del CSV y alertas adicionales de equipo/fecha, está en `docs/data_audit/`.


In [ ]:
# La primera pasada lee solo fechas; no se interpretan resultados posteriores.
matches = load_clean_matches(RAW, through_year=2023)
print('Filas limpias:', len(matches))
display(matches.head(3))
display(matches.dtypes.to_frame(name='tipo'))
print('Distribucion de la clase (hasta 2023):')
display(
    matches['winner']
    .value_counts()
    .reindex(CLASSES)
    .rename('cantidad')
    .to_frame()
)

## 3. Atributos causales

`build_causal_match_features` construye, para cada partido, tasas historicas calculadas con fechas estrictamente anteriores. Los partidos de un mismo dia se "featurizan" todos antes de actualizar los historiales, de modo que un partido jamas usa el resultado de otro partido del mismo dia (esto evita leakage).

El modelo usa 6 tasas de victoria:

- `home/away_win_rate_last_5`: forma de cada equipo en sus ultimos 5 partidos;
- `home/away_win_rate_season`: exito de cada equipo dentro del ano calendario;
- `home_win_rate_as_home_all`: historico del local actuando de local;
- `home_win_rate_h2h_as_home`: historico del local contra ese visitante con la misma localia (head-to-head orientado).

Cuando un equipo **no tiene historial** (debut, primera vez en la ventana o primer partido del ano) el denominador es 0 y se imputa un neutro de `0.5` en lugar de `0.0`, para separarlo de cero victorias; el neutro puede coincidir con una tasa real de 0.5.

El baseline usa las tasas causales de diez años del **mismo constructor**, con
los mismos partidos admitidos y el mismo valor neutro. Todos los modelos
pueden usar resultados de fechas anteriores de validación/test; ninguno usa
resultados de la fecha actual y ninguno se reajusta dentro de ese bloque.

In [ ]:
featured = build_causal_match_features(matches)
print('Partidos con atributos:', len(featured))
display(featured[['date', 'home', 'away'] + NUMERIC_FEATURES + ['winner']].head(5))

sin_historial = featured[featured['home_win_rate_season'].eq(0.5)]
print('Tasa 0.5: puede ser imputada o una proporción real, no prueba ausencia de historial:')
display(sin_historial[['date', 'home', 'away'] + NUMERIC_FEATURES].head(3))

## 4. Partición temporal y folds comunes

- Entrenamiento final: hasta 2023 inclusive.
- Evaluación reservada: 2024–2025.
- Validación: años completos 2021, 2022 y 2023, entrenando con todos los años
  anteriores a cada bloque. No se separan partidos de una misma fecha.

`CV_FOLDS` es la única definición de folds para ID3, NB, baseline y comparadores.
Cada llamada a `evaluate_temporal_cv` ajusta un pipeline nuevo por fold, con
selección de entradas y, cuando corresponde, discretización aprendida solo en
ese entrenamiento. Las métricas de test no seleccionan parámetros.


In [ ]:
train = featured.reset_index(drop=True)
assert train['date'].dt.year.le(2023).all()
CV_FOLDS = make_temporal_folds(train)
print('Partidos disponibles para selección:', len(train))
print('No se construyen atributos ni se evalúa el período 2024–2025.')
print('Folds compartidos:')
display(pd.DataFrame([
    {'validacion': fold.validation_year,
     'train_hasta': train.iloc[list(fold.train_positions)]['date'].max().date(),
     'validacion_desde': train.iloc[list(fold.validation_positions)]['date'].min().date(),
     'validacion_hasta': train.iloc[list(fold.validation_positions)]['date'].max().date(),
     'n_train': len(fold.train_positions),
     'n_validacion': len(fold.validation_positions)}
    for fold in CV_FOLDS
]))


## 5. Discretización

ID3 y Naive Bayes categórico reciben códigos enteros. Las tasas de los últimos
cinco partidos usan cortes fijos `[0.3, 0.6]`; con menos de cinco antecedentes
se calcula la proporción sobre los disponibles. Las otras tasas usan terciles
aprendidos en entrenamiento. Los valores repetidos pueden producir tamaños
desiguales o menos de tres intervalos. No hay categoría exclusiva sin historial.

Esta celda ilustra las seis tasas originales y conserva el diagnóstico ID3.
NB termina usando diez atributos: agrega puntos y diferencia de gol por
partido de ambos equipos, con terciles en esas cuatro entradas. Sus pipelines
nuevos se ajustan en las secciones 8–10 y 14; no reutilizan este discretizador
ni comparten cuantiles entre folds.


In [ ]:
discretizer = new_discretizer()
discretizer.fit(train[NUMERIC_FEATURES])

print('Bordes de cada atributo (dos cortes -> tres bines):')
display(
    pd.DataFrame(
        {column: discretizer.numeric_edges_[column] for column in NUMERIC_FEATURES}
    ).T.rename(columns={0: 'borde_1', 1: 'borde_2'})
)

X_train = discretizer.transform(train[NUMERIC_FEATURES])

print('Ejemplo: la tasa cruda y el codigo que recibe ID3 (1=baja, 2=media, 3=alta):')
muestra = train[['home', 'away'] + NUMERIC_FEATURES].head(5).copy()
all_columns = [column for column in NUMERIC_FEATURES]
codigos = pd.DataFrame(
    X_train[:5], columns=[column + '_cod' for column in NUMERIC_FEATURES]
)
display(pd.concat([muestra, codigos], axis=1))

print('Cuantas filas de train cae en cada categoria:')
resumen = pd.DataFrame({column: pd.Series(X_train[:, i]).map(
    {1: 'baja', 2: 'media', 3: 'alta'}
).value_counts() for i, column in enumerate(NUMERIC_FEATURES)})
display(resumen.reindex(BIN_ORDER).fillna(0).astype(int))

## 6. Como decide el arbol: entropia y ganancia

ID3 elige el atributo que mas reduce la entropia (desorden) de la clase. La entropia de la clase en train es:

```text
H(Y) = -sum_c P(c) * log2(P(c))
```

que mide cuantos bits de informacion hacen falta para decir el resultado sabiendo solo la distribucion global. La ganancia de informacion de un atributo es la entropia menos la entropia condicional promedio tras partir por ese atributo.

En este paso calculamos la primera decision que tomaria el arbol mostrando la ganancia de cada atributo sobre todo train.

In [ ]:
parent_entropy = entropia(train['winner'])
print(f'Entropia de la clase en train: {parent_entropy:.4f} bits '
      '(maximo teorico ~1.585 bits para 3 clases equiprobables)')

gains = []
for index, column in enumerate(NUMERIC_FEATURES):
    conditional = 0.0
    for code in np.unique(X_train[:, index]):
        mask = X_train[:, index] == code
        conditional += mask.mean() * entropia(train['winner'][mask])
    gains.append({
        'atributo': column,
        'entropia_condicional': round(conditional, 4),
        'ganancia': round(parent_entropy - conditional, 4),
    })
gains_frame = pd.DataFrame(gains).sort_values('ganancia', ascending=False)
display(gains_frame.reset_index(drop=True))

mejor = gains_frame.iloc[0]
print(
    f'Primera division sobre {mejor["atributo"]} '
    f'(ganancia {mejor["ganancia"]:.4f} bits).'
)

## 7. Naive Bayes propio y referencia de scikit-learn

### 7.1. Implementación propia: m-estimate

Ahora usamos `MEstimateCategoricalNB` (`src/naive_bayes.py`) con los mismos partidos, inicialmente las seis tasas de `NUMERIC_FEATURES` y las clases `E`, `L`, `V`. Naive Bayes combina la frecuencia de cada clase con las probabilidades de los atributos y supone que estos son **independientes entre si dada la clase**.

Esta implementacion es **categorica**: requiere entradas discretas, codificadas como enteros no negativos. Reutilizamos la configuracion de `MixedTypeDiscretizer` de la seccion 5: tres bines, cortes fijos para las tasas de los ultimos cinco partidos y cuantiles para las restantes. El discretizador existente se ajusto solo con train; para validar necesitamos instancias nuevas ajustadas solo con el entrenamiento de cada fold.

El suavizado **m-estimate** evita probabilidades condicionales nulas:

\[
P(X_j=v \mid Y=c) = \frac{n_{jvc} + m\,p_{jv}}{n_c + m}
\]

Aqui `n_jvc` cuenta los partidos de clase `c` cuyo atributo `j` tiene codigo `v`, y `n_c` cuenta los partidos de esa clase. El prior `p_jv = 1 / K_j` es uniforme sobre los codigos posibles, incluido el `0` reservado para desconocidos. Por defecto `K_j=max(X_train[:,j])+1`; `min_categories` permite declarar un dominio mayor sin mirar validacion/test. En esta grilla ambos NB declaran cuatro codigos, aunque un bin no aparezca en train. `m > 0` es el peso total del prior: cuanto mayor es, mas se acercan las condicionales a la distribucion uniforme. La consigna no fija el prior; elegimos el uniforme, no las frecuencias marginales.

**Como leer `src/naive_bayes.py`, paso a paso:**

1. **`_validate_X`** comprueba que cada fila sea un partido y que los valores sean codigos enteros no negativos. Las tasas se discretizan antes de entrar al modelo; los codigos son categorias, no cantidades.
2. **`fit(X, y)`** aprende dos cosas: la frecuencia de cada resultado `P(c)` y una tabla por atributo con `P(codigo | c)`. `np.unique` convierte etiquetas en indices y `np.bincount` cuenta apariciones. En cada tabla, las filas son clases y las columnas son codigos.
3. **Suavizado:** si hay 10 partidos de clase `L`, un codigo aparece en 3 y hay 4 codigos posibles (incluido el `0`), con `m=2` la probabilidad es `(3 + 2/4) / (10 + 2) = 0.292`. Incluso un codigo sin apariciones recibe algo de probabilidad. Este suavizado se aplica a las tablas de atributos, no a `P(c)`.
4. **`_joint_log_likelihood`** puntua cada clase para un partido: parte de `log P(c)` y suma el log de la probabilidad de cada uno de sus codigos. Asi implementa el producto de Naive Bayes sin multiplicar numeros muy pequenos:

\[
\mathrm{puntaje}(c) = \log P(c) + \sum_j \log P(X_j=x_j \mid c).
\]

5. **Las salidas:** `predict` elige la clase de mayor puntaje; `predict_proba` usa `softmax` para convertir los puntajes en probabilidades que suman 1; `predict_log_proba` normaliza directamente en log con `logsumexp`, evitando tomar el logaritmo de una probabilidad redondeada a cero. Las columnas siguen el orden de `classes_`. Si llega un codigo fuera del dominio aprendido/declarado, se usa el codigo reservado `0`.

Como ejemplo didáctico de la etapa inicial de seis tasas, para `[3, 1, 2, 2, 3, 1]`, se busca cada codigo en la tabla de su atributo y se suman sus aportes para `E`, `L` y `V`. No se suma el valor numerico de los codigos: se suman los **logs de sus probabilidades**. Para elegir la clase no hace falta calcular `P(X)`, porque es el mismo divisor para las tres.

### 7.2. CategoricalNB y equivalencia

La referencia estima `(n_jvc + alpha)/(n_c + alpha*K_j)`.
Ambos pipelines reciben la misma discretización; `min_categories=4` declara
tres bines y cero reservado. Con prior uniforme de categorías, prior de clase
empírico, `fit_prior=True`, `class_prior=None` y `force_alpha=True`,
`alpha=m/4` hace equivalentes las fórmulas. Se comprueban las decisiones y
puntajes, admitiendo únicamente diferencias de redondeo.

La configuración final usa **diez atributos**, detallados en la sección 10,
con `m=0.1` y `alpha=0.025`. Las seis tasas corresponden a la etapa inicial
de selección y al ejemplo didáctico, no al modelo final.


## 8. Grillas y regla de selección

Los folds son los mismos años completos 2021, 2022 y 2023. Cada combinación
ajusta un pipeline nuevo por fold, sin usar los cuantiles de otro entrenamiento.
Se selecciona por **macro-F1 medio de validación**, con igual peso por año y
las clases fijas E/L/V. El error es `1 - accuracy`; el de entrenamiento es de
resustitución y sirve como diagnóstico, no como estimación de generalización.

- ID3: `min_info_gain = [0, 0.001, 0.005, 0.01, 0.02, 0.05]`.
- NB propio: `m = [0.1, 1, 10, 100, 1000]`, `min_categories=4`.
- CategoricalNB: `alpha = [0.025, 0.25, 2.5, 25, 250]`, `min_categories=4`.
- Random Forest: `max_depth = [4, 8, None]` y
  `min_samples_leaf = [50, 20, 5, 1]`; 300 árboles, semilla 42,
  `class_weight='balanced_subsample'`.

Ante empate exacto se elige el menor umbral/m/alpha; para el bosque, la menor
profundidad y luego la mayor hoja. Se conserva el bosque sin límite/hoja 1,
los dos DecisionTree con entropía (códigos y tasas) y el baseline de diez años.

Con el mismo dominio de cuatro códigos por atributo en ambos NB, el suavizado uniforme propio equivale a
`alpha=m/4`. Las grillas de ambos NB permiten comparar esa correspondencia;
no se espera que dos implementaciones de la misma fórmula sean necesariamente
dos métodos estadísticos distintos. Los pipelines declaran cuatro códigos en cada fold. En general se necesita `alpha_j=m/K_j`: un único `alpha` solo es equivalente si todos los atributos tienen igual `K_j`, el mismo prior de clase y la misma codificación de desconocidos. Ver [la revisión de NB](docs/naive_bayes.md).

La celda siguiente reproduce directamente los **30 ajustes de NB** desde
`train`, sin archivos intermedios. La experimentación general de los otros
modelos permanece en `scripts/run_validation.py`; sus resultados existentes
son opcionales para esta ejecución de NB.


In [ ]:
import json
from nb_selection import NB_MODELS, final_nb_pipelines
from model_selection import run_model_selection, plot_validation_curves
from feature_experiments import (
    build_experiment_features, run_feature_experiments, feature_variants, fixed_candidates,
)

# Solo se entrega a la selección el prefijo hasta 2023.
assert train.date.dt.year.le(2023).all()
nb_cv_details, nb_cv_summary, nb_cv_selected = run_model_selection(train, models=NB_MODELS)
assert len(nb_cv_details) == 30

NB_DELIVERY = ROOT / 'results/nb_delivery'
VALIDATION_DIR = ROOT / 'results/validation'
USE_SHARED_RESULTS = True  # Tablas ajenas opcionales; nunca son insumos de NB.

def with_optional_other_models(current, path):
    if USE_SHARED_RESULTS and path.is_file():
        historical = pd.read_csv(path)
        return pd.concat([historical.loc[~historical.model.isin(NB_MODELS)], current],
                         ignore_index=True)
    return current.copy()

cv_details = with_optional_other_models(nb_cv_details, VALIDATION_DIR / 'fold_metrics.csv')
cv_summary = with_optional_other_models(nb_cv_summary, VALIDATION_DIR / 'grid_summary.csv')
cv_selected = with_optional_other_models(nb_cv_selected, VALIDATION_DIR / 'selected.csv')
print('NB calculado en esta ejecución; otras filas, si existen, son históricas.')
display(cv_selected[['model', 'parameters', 'macro_f1_mean', 'macro_f1_std',
                     'validation_error_mean', 'train_error_mean']].round(6))
selected_params = {row.model: json.loads(row.parameters) for row in cv_selected.itertuples()}
id3_gain_elegido = selected_params.get('ID3', {}).get('min_info_gain')
rf_params_elegidos = selected_params.get('Random Forest')
nb_m_elegido = selected_params['NB propio']['m']
nb_alpha_elegido = selected_params['CategoricalNB']['alpha']
id3_cv = cv_summary.loc[cv_summary.model.eq('ID3')].copy()
id3_cv['min_info_gain'] = id3_cv.parameters.map(lambda p: json.loads(p)['min_info_gain'])
nb_cv = nb_cv_summary.loc[nb_cv_summary.model.eq('NB propio')].copy()
nb_cv['m'] = nb_cv.parameters.map(lambda p: json.loads(p)['m'])
nb_cv['macro_f1_medio'] = nb_cv['macro_f1_mean']


## 9. Curvas de error y selección de suavizado

Estas curvas corresponden a la búsqueda original sobre **seis tasas**, repetida
con el código actual. El error es **1 − accuracy**; cada punto es la media simple
de los años 2021, 2022 y 2023. La curva de entrenamiento mide resustitución.
Se selecciona por **macro-F1 de validación sin redondear**, no por error de train.

`m=0.1, 1, 10` empatan en macro-F1 medio (0.363482); el desempate declarado
elige **m=0.1**. Lo mismo ocurre con `alpha=0.025, 0.25, 2.5`, eligiéndose
**alpha=0.025**. El error medio de validación es 0.565633 y el de entrenamiento
0.516039 para ambos seleccionados. Suavizar más no mejora el macro-F1 en esta grilla.

Las curvas coinciden porque ambos usan prior de clase empírico, prior uniforme
de categorías y **K=4** por atributo: `(n+m/4)/(n_c+m)` equivale a
`(n+alpha)/(n_c+4*alpha)` con `alpha=m/4`. Se declara K=4 en cada fold y variante,
incluido el cero reservado; `min_categories=4` conserva bines ausentes.
Esto no implica equivalencia con cualquier discretización: si los K_j difieren,
un solo alpha no equivale al mismo m para todos los atributos.

Las desviaciones entre tres años son descriptivas. Los folds usados para elegir
no constituyen una evaluación independiente ni permiten anticipar resultados de test.


In [ ]:
from IPython.display import Image, display

# Las figuras NB se generan ahora, no se leen de una corrida anterior.
for path in plot_validation_curves(nb_cv_summary, NB_DELIVERY / 'figures'):
    display(Image(filename=str(path)))
for model_name in ['ID3', 'NB propio', 'CategoricalNB', 'Random Forest']:
    part = cv_summary.loc[cv_summary.model.eq(model_name)]
    if not part.empty:
        print(model_name)
        display(part[['parameters', 'train_error_mean', 'validation_error_mean',
                      'train_macro_f1_mean', 'macro_f1_mean', 'macro_f1_std']].round(6))
for filename in ['id3.png', 'random_forest.png']:
    path = VALIDATION_DIR / 'figures' / filename
    if USE_SHARED_RESULTS and path.is_file():
        display(Image(filename=str(path)))
print('Detalle anual de las configuraciones elegidas:')
display(cv_details.loc[cv_details.config_id.isin(cv_selected.config_id),
    ['model', 'parameters', 'validacion', 'train_error', 'validation_error', 'macro_f1']])


## 10. Experimentos acotados de atributos

Se comparan siete variantes declaradas antes de la corrida: actuales; +puntos;
+diferencia de gol; +ambos; +tasas históricas de empate; sin historial local;
sin historial H2H del local. Los NB mantienen los hiperparámetros
seleccionados anteriormente; las filas de otros modelos se conservan como referencia histórica. Cada variante usa los mismos folds anuales y un
preprocesamiento nuevo por fold. Las tasas de empate solo usan fechas anteriores.

El diagnóstico «nunca E» restringe las predicciones a L/V, usando probabilidades
(o conteos del nodo en ID3). Se conservan todos los empates reales y las tres
clases E/L/V al calcular macro-F1; no se entrena un clasificador binario.
El diagnóstico no participa en la selección principal de atributos.

Selección: macro-F1 medio anual; empate exacto -> menos atributos, luego orden
del plan. Reutilizar validación para decisiones sucesivas puede introducir
optimismo. La sección 14 reproduce el reajuste y la evaluación final de NB.

Plan: `docs/feature_experiment_plan.md`. Resultados: `docs/feature_findings.md`.
Reproducción: `python3.12 scripts/run_feature_experiments.py`.

Para NB se repitieron las siete variantes con los hiperparámetros ya elegidos.
Ambos seleccionan **plus_points_and_goals**: macro-F1 medio **0.385798**
(DE 0.033246), frente a 0.363482 con las seis tasas. Accuracy baja de
0.434367 a **0.427650**; se conserva la variante por el criterio de macro-F1.
El macro-F1 por año es 0.373586, 0.431225 y 0.352584. El error medio de train
es 0.526476 y el de validación 0.572350 con las diez columnas.

Es una selección secuencial: primero suavizado con seis tasas, luego atributos
manteniendo ese suavizado; **no se retocó m/alpha con diez atributos**. El
diagnóstico nunca-E no puede ganar. La celda siguiente reproduce los **42 ajustes de NB** sin resultados precalculados.


In [ ]:
# Construye también las tasas de empate requeridas por una variante diagnóstica.
nb_experiment_frame = build_experiment_features(matches)
nb_feature_details, nb_feature_summary, nb_feature_selected, nb_validation_predictions = (
    run_feature_experiments(nb_experiment_frame, nb_cv_selected, models=NB_MODELS))
assert len(nb_feature_details) == 84  # 42 ajustes, dos decisiones por ajuste.
FEATURE_RESULTS = ROOT / 'results/feature_experiments'
feature_summary = with_optional_other_models(nb_feature_summary, FEATURE_RESULTS / 'summary.csv')
feature_selected = with_optional_other_models(nb_feature_selected, FEATURE_RESULTS / 'selected.csv')
print('NB calculado ahora; otros modelos, si existen, conservan su corrida histórica.')
display(feature_selected[['model', 'variant', 'parameters', 'accuracy_mean',
                          'macro_f1_mean', 'macro_f1_std']].round(6))
display(nb_feature_summary.loc[nb_feature_summary.decision.eq('three_class'),
    ['model', 'variant', 'parameters', 'accuracy_mean', 'macro_f1_mean', 'macro_f1_std']].round(6))
paired = feature_summary.pivot(index=['model', 'variant'], columns='decision',
                              values=['accuracy_mean', 'macro_f1_mean'])
for metric in ['accuracy_mean', 'macro_f1_mean']:
    paired[(metric, 'delta_never_draw')] = paired[(metric, 'never_draw')] - paired[(metric, 'three_class')]
print('Diagnóstico sin E (no elegible), conservando las tres clases verdaderas:')
display(paired.round(6))
path = FEATURE_RESULTS / 'accuracy_macro_f1.png'
if USE_SHARED_RESULTS and path.is_file():
    display(Image(filename=str(path)))


### Configuración NB definitiva, fijada antes del test

| Método | Configuración | Atributos | Decisión |
|---|---|---|---|
| NB propio | `m=0.1`, `min_categories=4` | `plus_points_and_goals` (10) | Tres clases E/L/V |
| CategoricalNB | `alpha=0.025`, `min_categories=4`, `force_alpha=True`, `fit_prior=True`, `class_prior=None` | La misma variante (10) | Tres clases E/L/V |

Columnas exactas, en el orden de entrada:

1. `home_win_rate_last_5`
2. `away_win_rate_last_5`
3. `home_win_rate_season`
4. `away_win_rate_season`
5. `home_win_rate_as_home_all`
6. `home_win_rate_h2h_as_home`
7. `home_points_per_match_5`
8. `away_points_per_match_5`
9. `home_goal_diff_per_match_5`
10. `away_goal_diff_per_match_5`

Puntos y diferencia de gol son **promedios por partido** de hasta cinco antecedentes
de cada equipo: puntos 3/1/0 y goles a favor menos goles en contra. Sin antecedentes,
se mantienen los neutros declarados (4/3 puntos, 0 de diferencia). Las dos tasas
recientes usan cortes fijos 0.3/0.6; las otras ocho columnas usan terciles aprendidos
solo con entrenamiento, al igual que sus medianas.

La celda siguiente obtiene parámetros y columnas de la validación recién calculada y crea
los pipelines **sin ajustarlos**. La sección 14 realiza el reajuste con todos los partidos hasta 2023 y
presenta la evaluación final, sin modificar esta selección.


In [ ]:
# Cerrar la configuración antes de leer los resultados de 2024–2025.
nb_fixed = {c.model: c for c in fixed_candidates(nb_cv_selected, models=NB_MODELS)}
nb_variants = feature_variants()
NB_FINAL_CONFIG = {
    row.model: {
        'estimator': type(nb_fixed[row.model].estimator).__name__,
        'estimator_parameters': nb_fixed[row.model].estimator.get_params(),
        'variant': row.variant, 'columns': list(nb_variants[row.variant]),
        'representation': 'discrete', 'decision': 'three_class',
        'preprocessing': {'n_bins': 3,
                          'fixed_cuts': {c: [0.3, 0.6] for c in NUMERIC_FEATURES[:2]},
                          'quantiles_and_medians': 'fit on training only'},
        'validation_macro_f1_mean': row.macro_f1_mean,
        'validation_accuracy_mean': row.accuracy_mean,
    } for row in nb_feature_selected.itertuples()
}
# Comprobar que la reproducción conserva la configuración ya cerrada.
assert nb_m_elegido == 0.1 and nb_alpha_elegido == 0.025
assert all(c['variant'] == 'plus_points_and_goals' for c in NB_FINAL_CONFIG.values())
nb_selection = {'final_configurations': NB_FINAL_CONFIG}
nb_config_before_test = json.dumps(NB_FINAL_CONFIG, sort_keys=True)
NB_FINAL_PIPELINES = final_nb_pipelines(nb_selection)
NB_FINAL_COLUMNS = tuple(NB_FINAL_CONFIG['NB propio']['columns'])
assert NB_FINAL_COLUMNS == tuple(NB_FINAL_CONFIG['CategoricalNB']['columns'])
assert len(NB_FINAL_COLUMNS) == 10
for name, pipeline in NB_FINAL_PIPELINES.items():
    assert not hasattr(pipeline.named_steps['model'], 'classes_')
    print(name, pipeline.named_steps['model'].get_params())
print('Entradas finales:', NB_FINAL_COLUMNS)
print('Configuración cerrada con validación hasta 2023; todavía no se ha leído test.')


## 14. Evaluación final de ambos NB

### 14.1. Reajuste con todo el histórico hasta 2023

Cada NB usa un pipeline nuevo con los diez atributos de `NB_FINAL_COLUMNS`.
Se ajustan sus medianas y terciles y el clasificador con los 14.705 partidos
admitidos hasta 2023 inclusive; las tasas recientes conservan cortes 0.3/0.6.
Los parámetros `m=0.1`, `alpha=0.025`, K=4 y la decisión de tres clases están
fijados por la selección anterior. No se reutiliza un discretizador de seis tasas.

El constructor causal actualiza historiales al terminar cada fecha. Durante test
permite resultados de fechas anteriores del propio test; nunca del día actual.
Ni discretizadores ni clasificadores se reajustan dentro de 2024–2025.

La celda ejecuta el ajuste y la predicción directamente desde el ZIP original.
Los resultados quedan en memoria; no carga modelos ni tablas precalculadas.
La configuración se comprueba antes y después del test. Los artefactos de la
evaluación anterior se conservan como evidencia, pero no son un requisito.


In [ ]:
from nb_final import evaluate_final_nb

nb_all_matches = load_clean_matches(RAW)
nb_all_features = build_causal_match_features(nb_all_matches)
nb_train_again, nb_test = temporal_holdout(nb_all_features)
pd.testing.assert_frame_equal(train, nb_train_again)
nb_final = evaluate_final_nb(nb_all_features, nb_selection)
assert json.dumps(NB_FINAL_CONFIG, sort_keys=True) == nb_config_before_test
nb_final['configurations'] = {'naive_bayes': NB_FINAL_CONFIG}
nb_final['manifest'] = {
    'python': sys.version.split()[0], 'packages': {'scikit-learn': sklearn.__version__},
    'split': {name: {'rows': len(part), 'start': str(part.date.min().date()),
                     'end': str(part.date.max().date())}
              for name, part in [('train', nb_train_again), ('test', nb_test)]},
}
# Fechas ISO uniformes para unir marcadores y mostrar ejemplos reproducibles.
nb_final['predictions']['date'] = nb_final['predictions'].date.dt.strftime('%Y-%m-%d')
print('Entorno:', nb_final['manifest']['python'], 'scikit-learn', sklearn.__version__)
display(pd.DataFrame(nb_final['manifest']['split']).T[['rows', 'start', 'end']])
for name in NB_MODELS:
    fitted_pre = nb_final['preprocessing'][name]
    assert fitted_pre['columns'] == list(NB_FINAL_COLUMNS)
    assert fitted_pre['state_before_test'] == fitted_pre['state_after_test']
    print(name, '— cortes ajustados hasta', fitted_pre['fitted_through'])
    display(pd.DataFrame({
        'atributo': fitted_pre['columns'],
        'cortes': [fitted_pre['numeric_edges'][c] for c in fitted_pre['columns']],
        'mediana': [fitted_pre['numeric_medians'][c] for c in fitted_pre['columns']],
        'K': fitted_pre['n_categories'],
    }))


### 14.2. Evaluación sobre los mismos 472 partidos de 2024–2025

Todas las métricas usan las predicciones finales de esta ejecución con la
configuración cerrada antes del test. Accuracy y macro-F1 se calculan sobre el bloque
completo; macro-F1 promedia E/L/V con igual peso y `zero_division=0`.
El último partido disponible es del 30/06/2025: no se evalúa todo 2025.

Las matrices usan filas reales y columnas predichas, siempre en orden E, L, V.
El baseline usa el mismo historial causal, ventana `[fecha − 10 años, fecha)`,
neutro 0.5 y empate de tasas a favor de L. Su evaluación histórica no acredita
la política actual, por lo que se recalculó sobre esta misma muestra.


In [ ]:
print('Métricas finales:')
display(nb_final['summary'].round(6))
print('Precision, recall, F1 y soporte — E, L, V:')
display(nb_final['class_report'].round(6))
from io import BytesIO
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), layout='constrained')
for axis, (name, matrix) in zip(axes, nb_final['confusion_matrices'].items()):
    print(name, '(filas reales; columnas predichas)')
    display(matrix)
    axis.imshow(matrix.to_numpy(), cmap='Blues', vmin=0, vmax=121)
    axis.set(xticks=range(3), xticklabels=CLASSES, yticks=range(3), yticklabels=CLASSES,
             xlabel='Predicción', ylabel='Real', title=name)
    for r in range(3):
        for c in range(3):
            axis.text(c, r, str(matrix.iloc[r, c]), ha='center', va='center',
                      color='white' if matrix.iloc[r, c] > 60 else 'black')
buffer = BytesIO()
fig.savefig(buffer, format='png', dpi=130)
display(Image(data=buffer.getvalue()))
plt.close(fig)


### 14.3. Ejemplos concretos y criterio reproducible

Se ordenan las predicciones finales por `date, home, away`, ascendentemente,
y se toma el primer partido de cada par `(winner, pred_nb_propio)` en orden
E/L/V. Así se cubren las nueve celdas no vacías de la matriz, incluidos tres
aciertos y seis errores. No se eligen por equipos conocidos, marcador o
confianza. Es una selección ilustrativa por tipo de resultado, no una muestra
representativa de sus frecuencias; favorece fechas tempranas.

Los marcadores se unen desde el ZIP original tras aplicar la misma política
de limpieza, mediante fecha y equipos, con correspondencia uno a uno y
verificación de la clase real. Se usan únicamente para describir los ejemplos.

| Fecha | Local | Visitante | Marcador real (L-V) | Real | NB (ambos) | Base |
|---|---|---|:---:|:---:|:---:|:---:|
| 2024-03-11 | Montevideo Wanderers | Deportivo Maldonado | 0-0 | E | E | L |
| 2024-02-25 | Cerro Largo FC | CA Fenix | 0-0 | E | L | L |
| 2024-02-18 | CA Cerro | Montevideo Wanderers | 1-1 | E | V | V |
| 2024-04-07 | CA Cerro | Rampla Juniors Futbol Club | 3-0 | L | E | L |
| 2024-02-17 | Nacional | River Plate | 2-1 | L | L | L |
| 2024-03-02 | River Plate | Danubio | 1-0 | L | V | L |
| 2024-02-24 | Montevideo Wanderers | Racing Club | 0-1 | V | E | L |
| 2024-02-18 | Deportivo Maldonado | Boston River | 1-2 | V | L | L |
| 2024-02-17 | CA Fenix | Danubio | 1-2 | V | V | V |

Wanderers–Maldonado ilustra un empate recuperado por NB que el baseline pierde.
Cerro–Rampla ilustra el costo opuesto: NB predice empate y el baseline acierta
la victoria local. Cerro Largo–Fenix y Cerro–Wanderers ejemplifican los dos
errores predominantes, E→L y E→V. Estos casos muestran decisiones observadas;
ni el marcador ni estos ejemplos explican por sí solos por qué se produjeron.


In [ ]:
# Unión descriptiva de las predicciones recién calculadas; sin ajustes adicionales.
nb_predicciones = nb_final['predictions'].copy()
assert len(nb_predicciones) == nb_final['manifest']['split']['test']['rows']
assert nb_predicciones.pred_nb_propio.eq(nb_predicciones.pred_categorical_nb).all()
nb_claves = ['date', 'home', 'away']
nb_marcadores = load_clean_matches(RAW)[nb_claves + ['winner', 'gh', 'ga']].copy()
nb_marcadores['date'] = nb_marcadores.date.dt.strftime('%Y-%m-%d')
nb_casos = nb_predicciones.merge(
    nb_marcadores.rename(columns={'winner': 'winner_original'}),
    on=nb_claves, how='left', validate='one_to_one', indicator=True)
assert nb_casos['_merge'].eq('both').all()
assert nb_casos.winner.eq(nb_casos.winner_original).all()
nb_casos['marcador_real'] = nb_casos.gh.astype(str) + '-' + nb_casos.ga.astype(str)
nb_ejemplos = (
    nb_casos.sort_values(nb_claves, kind='stable')
    .drop_duplicates(['winner', 'pred_nb_propio'], keep='first')
    .sort_values(['winner', 'pred_nb_propio'], kind='stable'))
display(nb_ejemplos[nb_claves + ['marcador_real', 'winner', 'pred_nb_propio',
                                 'pred_categorical_nb', 'pred_baseline_10y']])
print('Criterio: primer partido por fecha/local/visitante de cada par real/predicción.')


### 14.4. Equivalencia de ambos NB y comparación con el baseline

Se comprueban las predicciones una a una, además de las entradas discretas,
cardinalidades, conteos, priors y puntajes. Con K=4 en los diez atributos,
`(n + m/4)/(n_c + m)` coincide con `(n + alpha)/(n_c + 4*alpha)` para
`alpha=m/4=0.025`. Las diferencias de punto flotante se registran sin cambiar
la decisión por argmax ni el orden de desempate E/L/V.

`nb_final['disagreements']` conserva cualquier discrepancia con identidad del
partido, puntajes de cada modelo y márgenes entre las dos clases mejor puntuadas.


In [ ]:
nb_equivalence = nb_final['diagnosis']
print('Predicciones diferentes:', nb_equivalence['n_differences'],
      'de', nb_equivalence['n_predictions'])
print(nb_equivalence['explanation'])
print('Máxima diferencia absoluta de puntajes logarítmicos:',
      nb_equivalence['max_abs_joint_log_likelihood_difference'])
if nb_equivalence['n_differences']:
    display(nb_final['disagreements'])
nb_comparacion = nb_final['summary'].set_index('model')[['accuracy', 'macro_f1']]
nb_comparacion['delta_accuracy_baseline'] = nb_comparacion.accuracy - nb_comparacion.loc['Base 10 años', 'accuracy']
nb_comparacion['delta_macro_f1_baseline'] = nb_comparacion.macro_f1 - nb_comparacion.loc['Base 10 años', 'macro_f1']
display(nb_comparacion.round(6))


### 14.5. Análisis cuantitativo y comparación

E significa empate, L victoria local y V victoria visitante. El test contiene
472 partidos admitidos del 17/02/2024 al 30/06/2025 (297 de 2024 y 175 de
2025; este último año está incompleto). Ambos NB coinciden en las 472
predicciones; por eso el análisis de NB vale para las dos implementaciones.

| Modelo | Aciertos / total | Accuracy | Macro-F1 |
|---|---:|---:|---:|
| NB propio / CategoricalNB | 226/472 | 0.4788 | 0.4454 |
| Baseline de diez años | 218/472 | 0.4619 | 0.3564 |

| Clase | Soporte | Precision NB | Recall NB | F1 NB | Precision base | Recall base | F1 base |
|:---:|---:|---:|---:|---:|---:|---:|---:|
| E | 131 | 0.3537 | 0.2214 | 0.2723 | 0.0000 | 0.0000 | 0.0000 |
| L | 190 | 0.5475 | 0.6368 | 0.5888 | 0.5021 | 0.6368 | 0.5615 |
| V | 151 | 0.4497 | 0.5033 | 0.4750 | 0.4199 | 0.6424 | 0.5079 |

L tiene las mejores precision, recall y F1; V queda en segundo lugar y E
en último. NB predice 82 empates (17.37% del test), aunque hay 131 reales
(27.75%): acierta 29, omite 102 y genera 53 falsos empates. El baseline
no predice E; su precision indefinida para E se informa como cero
(`zero_division=0`). Macro-F1 da igual peso a las tres clases.

Matrices de confusión: filas reales y columnas predichas, en orden E/L/V.

| Real | NB: E | NB: L | NB: V | Base: E | Base: L | Base: V |
|:---:|---:|---:|---:|---:|---:|
| E | 29 | 54 | 48 | 0 | 66 | 65 |
| L | 24 | 121 | 45 | 0 | 121 | 69 |
| V | 29 | 46 | 76 | 0 | 54 | 97 |

De los 246 errores NB, predominan los empates clasificados como victorias:
102 (41.46%), repartidos entre E→L (54, la celda fuera de la diagonal más
frecuente) y E→V (48). Las confusiones L→V (45) y V→L (46) suman 91
(36.99%); las victorias clasificadas como empate suman 53 (21.54%).

NB supera al baseline en 1.69 puntos porcentuales de accuracy y 0.0889 de
macro-F1. Ambos aciertan 153 partidos y ambos fallan 181; NB corrige 73
errores del baseline, pero pierde 65 aciertos de este: saldo +8.
Por clase, NB recupera 29 empates adicionales, conserva 121 aciertos locales
y baja de 97 a 76 aciertos visitantes. El beneficio no es uniforme:
mejoran F1 de E y L, pero empeora F1 de V. Son diferencias descriptivas;
no se evaluó su significación estadística.


In [ ]:
# Recalcular métricas desde las predicciones finales y contrastar las tablas.
from sklearn.metrics import precision_recall_fscore_support

nb_etiquetas = ['E', 'L', 'V']
for nombre, columna, correcto in [
    ('NB propio', 'pred_nb_propio', 'correct_nb_propio'),
    ('CategoricalNB', 'pred_categorical_nb', 'correct_categorical_nb'),
    ('Base 10 años', 'pred_baseline_10y', 'correct_baseline_10y'),
]:
    real, pred = nb_predicciones.winner, nb_predicciones[columna]
    assert real.eq(pred).eq(nb_predicciones[correcto]).all()
    metricas = [accuracy_score(real, pred),
                f1_score(real, pred, labels=nb_etiquetas, average='macro', zero_division=0)]
    guardadas = nb_final['summary'].set_index('model').loc[nombre, ['accuracy', 'macro_f1']]
    np.testing.assert_allclose(metricas, guardadas.to_numpy(dtype=float))
    np.testing.assert_array_equal(
        confusion_matrix(real, pred, labels=nb_etiquetas),
        nb_final['confusion_matrices'][nombre].loc[nb_etiquetas, nb_etiquetas])
    por_clase = np.column_stack(precision_recall_fscore_support(
        real, pred, labels=nb_etiquetas, zero_division=0))
    tabla = nb_final['class_report'].query('model == @nombre').set_index('class')
    np.testing.assert_allclose(
        por_clase, tabla.loc[nb_etiquetas, ['precision', 'recall', 'f1', 'support']])

print('Comparación pareada (filas: acierto NB, columnas: acierto baseline):')
display(pd.crosstab(nb_predicciones.correct_nb_propio, nb_predicciones.correct_baseline_10y))
nb_errores = (nb_predicciones.loc[~nb_predicciones.correct_nb_propio]
              .groupby(['winner', 'pred_nb_propio']).size().rename('casos').reset_index())
nb_errores['porcentaje_de_errores'] = 100 * nb_errores.casos / nb_errores.casos.sum()
display(nb_errores.sort_values('casos', ascending=False))

# Un solo diagnóstico: signo de la diferencia de puntos recientes, sin umbral ajustado.
nb_atributos = nb_test[nb_claves + ['winner'] + list(NB_FINAL_COLUMNS)].copy()
nb_atributos['date'] = nb_atributos.date.dt.strftime('%Y-%m-%d')
nb_escenarios = nb_predicciones.merge(
    nb_atributos, on=nb_claves + ['winner'], how='left', validate='one_to_one', indicator=True)
assert nb_escenarios['_merge'].eq('both').all()
nb_diferencia = nb_escenarios.home_points_per_match_5 - nb_escenarios.away_points_per_match_5
assert np.isfinite(nb_diferencia).all()
nb_escenarios['escenario'] = np.select(
    [nb_diferencia.gt(0), nb_diferencia.lt(0)], ['Local > visitante', 'Local < visitante'],
    default='Local = visitante')
nb_resumen_escenarios = []
for escenario in ['Local > visitante', 'Local = visitante', 'Local < visitante']:
    grupo = nb_escenarios.loc[nb_escenarios.escenario.eq(escenario)]
    nb_resumen_escenarios.append({
        'escenario': escenario, 'casos': len(grupo),
        'reales_E_L_V': '/'.join(str(int(grupo.winner.eq(c).sum())) for c in nb_etiquetas),
        'aciertos_NB': int(grupo.correct_nb_propio.sum()),
        'accuracy_NB': grupo.correct_nb_propio.mean(),
        'macro_f1_NB': f1_score(grupo.winner, grupo.pred_nb_propio,
                                labels=nb_etiquetas, average='macro', zero_division=0),
        'aciertos_base': int(grupo.correct_baseline_10y.sum()),
        'accuracy_base': grupo.correct_baseline_10y.mean(),
    })
nb_resumen_escenarios = pd.DataFrame(nb_resumen_escenarios)
assert nb_resumen_escenarios.casos.sum() == len(nb_predicciones)
assert nb_resumen_escenarios.aciertos_NB.sum() == nb_predicciones.correct_nb_propio.sum()
display(nb_resumen_escenarios.round(4))
print('Diagnóstico descriptivo posterior al test; no modifica la configuración cerrada.')


### 14.6. Escenario observable antes del partido

Como diagnóstico posterior a la evaluación, se particiona **todo** el test por
el signo de `home_points_per_match_5 - away_points_per_match_5`: mayor,
igual o menor que cero. Son los promedios calculados de hasta cinco partidos
anteriores de cada equipo, no necesariamente cinco partidos consecutivos del
calendario. No se buscan umbrales ni se escoge el mejor entre varios cortes.
La igualdad usa exactamente los valores calculados en memoria, sin redondear. La partición es exhaustiva y disjunta.

| Puntos recientes | Casos | Reales E/L/V | Aciertos NB | Accuracy NB | Aciertos base | Accuracy base |
|---|---:|:---:|---:|---:|---:|---:|
| Local > visitante | 216 | 62/106/48 | 116 | 53.70% | 99 | 45.83% |
| Local = visitante | 39 | 9/15/15 | 15 | 38.46% | 21 | 53.85% |
| Local < visitante | 217 | 60/69/88 | 95 | 43.78% | 98 | 45.16% |

**Hecho observado:** NB tiene mayor accuracy cuando el local presenta mayor
promedio reciente: 116/216, frente a 95/217 con ventaja visitante y 15/39 con
igualdad. Frente al baseline gana 17 aciertos en el primer grupo y pierde 6 y
3 en los restantes. La diferencia entre ventaja local e igualdad es de
15.24 puntos porcentuales, pero el grupo de igualdad tiene solo 39 casos.

**Posible explicación, no comprobada:** la ventaja reciente local podría ser
una señal más útil para estas predicciones. También cambia la composición
de clases: hay 106 victorias locales entre los 216 casos con ventaja local,
frente a 15 entre los 39 con igualdad, y L es la clase mejor reconocida.
No se aisló el efecto de la forma reciente respecto de esa composición ni de
los demás atributos. El mayor accuracy no implica una mejora uniforme:
el macro-F1 NB de estos grupos es 0.3788, 0.3737 y 0.3741, respectivamente.
Estas observaciones no constituyen una regla nueva de predicción, ni se usaron
para seleccionar modelos, atributos, cortes o hiperparámetros.


### 14.7. Limitaciones y alcance de las explicaciones

- **Discretización:** los dos cortes fijos 0.3/0.6 de las tasas recientes y
  los terciles aprendidos en train para los otros ocho atributos reducen
  números a tres bines (más el código cero reservado). Valores distintos
  dentro de un bin resultan indistinguibles y valores próximos a un corte
  pueden separarse. Por ejemplo, tasas recientes 0.0 y 0.2 comparten bin.
  Esto describe pérdida de resolución; no demuestra que produzca los
  errores listados. No se compararon nuevos cortes con test.
- **Atributos históricos:** los promedios resumen ventanas de diferente
  extensión y omiten parte del contexto del partido. Cinco antecedentes
  pueden cubrir lapsos distintos; el historial completo puede mezclar
  épocas y el de enfrentamientos puede tener soportes distintos. Las diez
  entradas no incluyen sus denominadores, por lo que tasas iguales no
  distinguen la cantidad de evidencia que las respalda. La tasa neutra 0.5
  también puede ser una proporción observada (por ejemplo, una victoria en
  dos partidos): **0.5 no demuestra falta de historial**. Para atribuir un
  caso a esa ausencia harían falta los conteos previos de la ventana y
  equipo correspondientes; aquí no se infiere esa condición.
- **Independencia condicional:** NB multiplica las probabilidades de los
  atributos dada E, L o V. Tasas de victorias, puntos y diferencia de gol
  resumen partidos que se superponen; esto hace plausible que compartan
  información y que tratarla como independiente distorsione los puntajes.
  La superposición por sí sola no prueba dependencia condicionada a la
  clase. No se midió esa dependencia ni se hizo una comparación que aísle
  su efecto; no se atribuye ningún error concreto a esta hipótesis.
- **Generalización:** los resultados describen estos 472 registros admitidos,
  con 2025 incompleto. Los historiales incluyen resultados de fechas
  estrictamente anteriores, incluso del test, y no se actualizan con el
  partido actual ni otros del mismo día. Es evaluación secuencial; no una
  predicción simultánea de todo 2024–2025 desde 2023. Los modelos y
  discretizadores permanecen fijos. Las diferencias de grupos y modelos
  no prueban causalidad ni garantizan desempeño futuro.

El análisis responde al apartado 3 de `Tarea_1.pdf` en el alcance solicitado:
NB propio, NB de scikit-learn y baseline. La evaluación pendiente de árboles
y Random Forest no se completa con estos resultados.


## Estado de la entrega

**NB completo:** carga del ZIP, limpieza y atributos causales; selección temporal
reproducible (30 + 42 ajustes); ambos modelos finales de diez atributos; baseline;
métricas, matrices, ejemplos y análisis. `informe.tex` integra estos resultados
en el documento común. La ejecución verificada se guarda mediante
`scripts/verify_final_nb.py` en `results/nb_delivery/notebook.executed.ipynb`.

**Pendientes compartidos:** las celdas de evaluación final de árboles permanecen
desactivadas. ID3 todavía usa seis tasas en esas celdas, aunque su comparación
de variantes seleccionó puntos y goles; falta integrar esa selección, ejecutar
su evaluación y la de Random Forest, completar las filas del informe y revisar
el resumen y las conclusiones conjuntas. Sin los CSV históricos faltan además
sus parámetros seleccionados; sus scripts de experimentación se conservan.
Faltan nombres/correos de autores y la confirmación personal de herramientas,
tareas y revisión humana para la declaración de IA. No se infieren esos datos.
